# 🧠 Brain MRI Tumor Segmentation using U-Net
**Author:** Samina Mazhar
**Dataset:** LGG Brain MRI Segmentation (Kaggle)
**Goal:** Pixel-level segmentation of brain tumors in MRI scans

## 📦 Step 1: Install & Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import os
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
print('TensorFlow version:', tf.__version__)
print('All libraries imported successfully! ✅')

## 📂 Step 2: Load Dataset

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 25

def load_data(image_dir, mask_dir):
    images = []
    masks = []
    for img_name in sorted(os.listdir(image_dir)):
        if img_name.endswith('.png') or img_name.endswith('.jpg'):
            # Load image
            img_path = os.path.join(image_dir, img_name)
            img = cv2.imread(img_path)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = img / 255.0
            images.append(img)
            # Load mask
            mask_path = os.path.join(mask_dir, img_name)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
            mask = mask / 255.0
            mask = np.expand_dims(mask, axis=-1)
            masks.append(mask)
    return np.array(images), np.array(masks)

print('Data loading function ready! ✅')

## 🔍 Step 3: Visualize Sample Data

In [ ]:
def visualize_samples(images, masks, n=5):
    fig, axes = plt.subplots(2, n, figsize=(20, 8))
    for i in range(n):
        axes[0, i].imshow(images[i])
        axes[0, i].set_title('MRI Scan')
        axes[0, i].axis('off')
        axes[1, i].imshow(masks[i].squeeze(), cmap='gray')
        axes[1, i].set_title('Tumor Mask')
        axes[1, i].axis('off')
    plt.suptitle('Sample MRI Scans with Tumor Masks', fontsize=16)
    plt.tight_layout()
    plt.savefig('../results/sample_visualization.png')
    plt.show()
    print('Visualization saved! ✅')

## 🏗️ Step 4: Build U-Net Architecture

In [ ]:
def conv_block(inputs, filters):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    return x

def encoder_block(inputs, filters):
    x = conv_block(inputs, filters)
    p = layers.MaxPooling2D()(x)
    return x, p

def decoder_block(inputs, skip, filters):
    x = layers.UpSampling2D()(inputs)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

def build_unet(input_shape=(256, 256, 3)):
    inputs = layers.Input(input_shape)
    # Encoder
    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)
    # Bridge
    b = conv_block(p4, 1024)
    # Decoder
    d1 = decoder_block(b, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)
    # Output
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(d4)
    model = Model(inputs, outputs)
    return model

model = build_unet()
model.summary()
print('U-Net model built successfully! ✅')

## 📊 Step 5: Define Metrics & Compile Model

In [ ]:
def dice_coefficient(y_true, y_pred):
    smooth = 1e-6
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (
        tf.keras.backend.sum(y_true_f) + 
        tf.keras.backend.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=dice_loss,
    metrics=['accuracy', dice_coefficient]
)
print('Model compiled successfully! ✅')

## 🚀 Step 6: Train Model

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        '../models/unet_best.h5',
        save_best_only=True,
        monitor='val_dice_coefficient',
        mode='max'
    ),
    tf.keras.callbacks.EarlyStopping(
        patience=10,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        factor=0.1,
        patience=5
    )
]
print('Callbacks ready! ✅')
print('Run this after loading your dataset:')
print("""
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)
""")

## 📈 Step 7: Plot Training Results

In [ ]:
def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    # Loss
    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_title('Model Loss')
    axes[0].legend()
    # Dice Coefficient
    axes[1].plot(history.history['dice_coefficient'], label='Train Dice')
    axes[1].plot(history.history['val_dice_coefficient'], label='Val Dice')
    axes[1].set_title('Dice Coefficient')
    axes[1].legend()
    plt.savefig('../results/training_history.png')
    plt.show()
    print('Training plots saved! ✅')

## 🎯 Step 8: Predict & Visualize Results

In [ ]:
def predict_and_visualize(model, images, masks, n=5):
    predictions = model.predict(images[:n])
    fig, axes = plt.subplots(3, n, figsize=(20, 12))
    for i in range(n):
        # Original MRI
        axes[0, i].imshow(images[i])
        axes[0, i].set_title('MRI Scan')
        axes[0, i].axis('off')
        # True Mask
        axes[1, i].imshow(masks[i].squeeze(), cmap='gray')
        axes[1, i].set_title('True Mask')
        axes[1, i].axis('off')
        # Predicted Mask
        axes[2, i].imshow(predictions[i].squeeze(), cmap='gray')
        axes[2, i].set_title('Predicted Mask')
        axes[2, i].axis('off')
    plt.suptitle('MRI Segmentation Results', fontsize=16)
    plt.tight_layout()
    plt.savefig('../results/predictions.png')
    plt.show()
    print('Predictions saved! ✅')

## ✅ Project Complete!
**Next Steps:**
- Download dataset from Kaggle
- Run this notebook in Google Colab
- Update results in README.md